[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day30_transformers/day30_notebook.ipynb)

# Day 30 / 42: Transformers
### #42DaysOfML | Week 4: Deep Learning

---

## What You'll Learn
- Why Transformers replaced RNNs (and what RNNs couldn't do)
- Self-attention from scratch — the math and the code
- Multi-head attention and why multiple heads matter
- Positional encoding — how Transformers track word order
- Build a minimal Transformer encoder from scratch
- Use HuggingFace BERT for text classification in 15 lines
- Production problem: context window limits in deployment

---

In [ ]:
!pip install torch transformers datasets matplotlib numpy --quiet

## The Concept

The 2017 paper "Attention Is All You Need" (Vaswani et al., Google Brain) introduced the Transformer architecture. It replaced recurrence entirely with **self-attention**: a mechanism that lets every token in a sequence attend to every other token simultaneously.

**Why RNNs fell short:**
- Sequential computation: token 10 can only be processed after tokens 1-9. This blocks parallelism.
- Long-range dependencies: even with LSTMs, information from token 1 must survive 100+ steps to influence token 100.
- Fixed-size hidden state: the entire sequence history is compressed into one vector.

**What self-attention does:**
Every token computes a weighted sum of all other tokens, where the weights are learned from the content of the tokens themselves. Token 50 can directly attend to token 1 with no intermediate steps.

**The three matrices — Q, K, V:**
- **Query (Q):** What am I looking for?
- **Key (K):** What do I have to offer?
- **Value (V):** What information do I carry?

For each token, you compute: `Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V`

The dot product QK^T measures compatibility between every query and every key. Dividing by sqrt(d_k) prevents extremely large values that would push softmax into saturation. The result is a probability distribution over all tokens, which you use to take a weighted sum of the values.

**Multi-head attention:**
Run self-attention h times in parallel with different learned projections. Each head can learn to attend to different aspects — one head might learn syntactic relationships, another semantic similarity, another coreference. Concatenate all heads' outputs.

**Positional encoding:**
Self-attention is permutation-invariant: it treats "cat sat mat" and "mat sat cat" identically without positional information. Positional encodings (sinusoidal functions or learned embeddings) are added to token embeddings to inject word order.

In [ ]:
# ============================================================
# SECTION 1: Self-Attention From Scratch
# Implement the exact equation from the Attention Is All You Need paper
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(Q, K, V):
    """
    Scaled dot-product attention.
    Q: (seq_len, d_k)
    K: (seq_len, d_k)
    V: (seq_len, d_v)
    Returns: (seq_len, d_v), attention_weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute dot product of queries and keys
    scores = Q @ K.T  # (seq_len, seq_len)
    
    # Step 2: Scale by sqrt(d_k) to prevent vanishing gradients in softmax
    scores = scores / np.sqrt(d_k)
    
    # Step 3: Softmax gives attention weights (probability distribution)
    weights = softmax(scores, axis=-1)  # (seq_len, seq_len)
    
    # Step 4: Weighted sum of values
    output = weights @ V  # (seq_len, d_v)
    
    return output, weights


# Simulate a short sentence: "the cat sat on the mat"
np.random.seed(42)
sentence = ["the", "cat", "sat", "on", "the", "mat"]
seq_len = len(sentence)
d_model = 8   # embedding dimension
d_k = 4       # key/query dimension
d_v = 4       # value dimension

# Simulate token embeddings (in practice, learned)
X = np.random.randn(seq_len, d_model)  # (6, 8)

# Projection matrices (in practice, these are learned linear layers)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

# Project embeddings to Q, K, V
Q = X @ W_Q  # (6, 4)
K = X @ W_K  # (6, 4)
V = X @ W_V  # (6, 4)

output, attn_weights = self_attention(Q, K, V)

# Visualise attention weights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels(sentence, fontsize=12)
axes[0].set_yticklabels(sentence, fontsize=12)
axes[0].set_xlabel('Key (what tokens are attended to)', fontsize=11)
axes[0].set_ylabel('Query (which token is attending)', fontsize=11)
axes[0].set_title('Attention Weight Matrix\n(darker = stronger attention)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0])

for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f'{attn_weights[i,j]:.2f}',
                    ha='center', va='center', fontsize=9,
                    color='white' if attn_weights[i,j] > 0.5 else 'black')

# Show step-by-step computation for 'cat'
cat_idx = 1  # index of 'cat'
scores_cat = (Q[cat_idx] @ K.T) / np.sqrt(d_k)
weights_cat = softmax(scores_cat)

colors = ['#4CAF50' if w > 0.2 else '#2196F3' if w > 0.1 else '#90CAF9' for w in weights_cat]
bars = axes[1].bar(sentence, weights_cat, color=colors, edgecolor='black', alpha=0.85)
for bar, w in zip(bars, weights_cat):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{w:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Attention Weight', fontsize=12)
axes[1].set_title(f'Attention Weights for Token: "{sentence[cat_idx]}"\n(how much "cat" attends to each word)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Input:   X shape = {X.shape}     (seq_len=6, d_model=8)")
print(f"Q, K, V shapes = {Q.shape}     (seq_len=6, d_k=4)")
print(f"Scores:  QK^T shape = ({seq_len},{seq_len})   (all-to-all similarity)")
print(f"Output:  shape = {output.shape}     (seq_len=6, d_v=4)")
print(f"\nAttention weights for 'cat' (sum to 1.0):")
for word, w in zip(sentence, weights_cat):
    bar = '█' * int(w * 30)
    print(f"  {word:6s}: {w:.4f}  {bar}")

In [ ]:
# ============================================================
# SECTION 2: Positional Encoding
# Sinusoidal encoding from the original Transformer paper
# ============================================================

def positional_encoding(max_len, d_model):
    """
    Sinusoidal positional encoding from 'Attention Is All You Need'.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len)[:, np.newaxis]      # (max_len, 1)
    dims = np.arange(0, d_model, 2)[np.newaxis, :]     # (1, d_model/2)
    
    div_term = np.power(10000, dims / d_model)
    
    PE[:, 0::2] = np.sin(positions / div_term)  # even dims: sin
    PE[:, 1::2] = np.cos(positions / div_term)  # odd dims: cos
    
    return PE


max_len = 50
d_model = 64
PE = positional_encoding(max_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of all positional encodings
im = axes[0].imshow(PE, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding Dimension', fontsize=12)
axes[0].set_ylabel('Position in Sequence', fontsize=12)
axes[0].set_title('Sinusoidal Positional Encodings\n(50 positions × 64 dimensions)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Show a few dimensions across positions
for dim in [0, 1, 4, 8, 16, 32]:
    axes[1].plot(PE[:, dim], label=f'dim {dim}', linewidth=2)
axes[1].set_xlabel('Position in Sequence', fontsize=12)
axes[1].set_ylabel('Encoding Value', fontsize=12)
axes[1].set_title('Positional Encoding Values Across Positions\n(different dims have different frequencies)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show that nearby positions have similar encodings
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pos = 10
print(f"Positional encoding shape: {PE.shape}")
print(f"\nSimilarity between positional encodings (position {pos} as reference):")
for other_pos in [9, 10, 11, 15, 20, 30, 49]:
    sim = cosine_similarity(PE[pos], PE[other_pos])
    print(f"  pos {pos:2d} vs pos {other_pos:2d}: similarity = {sim:.4f}")

print(f"\nKey property: nearby positions have higher similarity.")
print(f"The model can learn to use this to reason about relative position.")

In [ ]:
# ============================================================
# SECTION 3: Multi-Head Attention with PyTorch
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention from scratch.
    The same mechanism used in BERT, GPT, and all modern LLMs.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head
        
        # Single projection matrices that will be split across heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def split_heads(self, x):
        """Split d_model into num_heads independent attention heads."""
        batch, seq_len, d_model = x.shape
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        return x.permute(0, 2, 1, 3)  # (batch, num_heads, seq_len, d_k)
    
    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape
        
        # Project and split into heads
        Q = self.split_heads(self.W_Q(x))  # (batch, heads, seq_len, d_k)
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        
        # Scaled dot-product attention for all heads in parallel
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, heads, seq_len, seq_len)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # attention probabilities
        
        # Weighted sum of values
        attended = attn_weights @ V  # (batch, heads, seq_len, d_k)
        
        # Concatenate heads and project back to d_model
        attended = attended.permute(0, 2, 1, 3).contiguous()       # (batch, seq_len, heads, d_k)
        attended = attended.view(batch, seq_len, self.d_model)      # (batch, seq_len, d_model)
        
        output = self.W_O(attended)  # (batch, seq_len, d_model)
        return output, attn_weights


# Test multi-head attention
d_model = 64
num_heads = 8
seq_len = 10
batch = 2

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
x = torch.randn(batch, seq_len, d_model)
output, attn_weights = mha(x)

print("Multi-Head Attention Shapes")
print("=" * 45)
print(f"Input:          {list(x.shape)}         (batch, seq_len, d_model)")
print(f"Output:         {list(output.shape)}         (batch, seq_len, d_model)")
print(f"Attn weights:   {list(attn_weights.shape)}   (batch, heads, seq_len, seq_len)")
print(f"\nPer-head d_k:   {d_model // num_heads} (d_model / num_heads)")
print(f"Each of the {num_heads} heads attends to the same sequence with different learned projections.")
print(f"One head might learn 'who does what', another 'when', another 'subject-verb agreement'.")

# Verify attention weights sum to 1 across sequence for each head
sums = attn_weights.sum(dim=-1)
print(f"\nAttention weights sum to 1.0: {torch.allclose(sums, torch.ones_like(sums), atol=1e-5)}")

In [ ]:
# ============================================================
# SECTION 4: Full Transformer Encoder Block From Scratch
# ============================================================

class TransformerEncoderBlock(nn.Module):
    """
    One Transformer encoder block:
    LayerNorm -> MultiHeadAttention -> Residual
    LayerNorm -> FeedForward -> Residual
    
    This is the building block of BERT. Stack N of these = BERT encoder.
    BERT-base: N=12, d_model=768, num_heads=12, ffn_dim=3072
    """
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()
        
        self.attention = MultiHeadAttention(d_model, num_heads)
        
        # Feed-forward network: expand then contract
        # In BERT: d_model=768 -> ffn_dim=3072 -> d_model=768 (4x expansion)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),  # GELU used in BERT; ReLU in original Transformer
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model)
        )
        
        # Layer normalisation (applied before attention and FFN in modern Transformers)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm variant (used in GPT-2, modern practice)
        # Sub-layer 1: Multi-head self-attention with residual connection
        attn_out, _ = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)  # residual connection
        
        # Sub-layer 2: Position-wise feed-forward with residual connection
        x = x + self.dropout(self.ffn(self.norm2(x)))  # residual connection
        return x


class MiniTransformerEncoder(nn.Module):
    """
    A minimal Transformer encoder for text classification.
    Architecture: Embedding + PE -> N x Encoder Blocks -> Pool -> Classify
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, ffn_dim,
                 max_len, num_classes, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, d_model)  # learned positional embeddings
        
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x):
        batch, seq_len = x.shape
        
        # Token embeddings + positional embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        tok_emb = self.embedding(x)
        pos_emb = self.pos_embedding(positions)
        x = self.dropout(tok_emb + pos_emb)
        
        # N transformer blocks
        for layer in self.layers:
            x = layer(x)
        
        x = self.norm(x)
        
        # Mean pooling over sequence (CLS token approach is the BERT way)
        pooled = x.mean(dim=1)  # (batch, d_model)
        return self.classifier(pooled)


# Create a small model and verify shapes
model = MiniTransformerEncoder(
    vocab_size=1000,
    d_model=64,
    num_heads=4,
    num_layers=2,
    ffn_dim=256,
    max_len=128,
    num_classes=2
)

dummy_input = torch.randint(0, 1000, (4, 32))  # batch=4, seq_len=32
dummy_output = model(dummy_input)

total_params = sum(p.numel() for p in model.parameters())
print("Mini Transformer Encoder")
print("=" * 45)
print(f"Input shape:    {list(dummy_input.shape)}       (batch=4, seq_len=32)")
print(f"Output shape:   {list(dummy_output.shape)}             (batch=4, num_classes=2)")
print(f"Total params:   {total_params:,}")
print(f"\nFor reference, BERT-base has 110M parameters.")
print(f"Our mini model: {total_params:,} params ({total_params/110e6*100:.4f}% of BERT-base)")

In [ ]:
# ============================================================
# SECTION 5: BERT for Sentiment Classification in 15 Lines
# This is how production teams actually use Transformers
# ============================================================
from transformers import pipeline

# HuggingFace pipeline abstracts tokenisation + model + post-processing
# distilbert is a 40% smaller, 60% faster version of BERT-base
# with 97% of BERT's performance on GLUE benchmarks
sentiment_pipeline = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1  # -1 = CPU, 0 = first GPU
)

# ML-relevant test sentences
test_texts = [
    "This model achieves state of the art results on all benchmark datasets.",
    "Training diverged completely and the loss never decreased.",
    "The neural network learned meaningful feature representations from limited data.",
    "Terrible performance on the test set despite weeks of hyperparameter tuning.",
    "Transfer learning dramatically improved our results with only 500 labeled examples.",
    "The model overfits severely and fails to generalize to any new input.",
    "Excellent convergence and stable training loss across all experiments.",
    "The deployment failed because the inference latency exceeded the SLA by 10x.",
]

results = sentiment_pipeline(test_texts)

print("BERT (DistilBERT) Sentiment Classification")
print("=" * 65)
for text, result in zip(test_texts, results):
    label = result['label']
    score = result['score']
    icon = '' if label == 'POSITIVE' else ''
    print(f"{icon} [{label:8s}] ({score:.3f}) {text[:60]}..." if len(text) > 60
          else f"{icon} [{label:8s}] ({score:.3f}) {text}")

print(f"\n15 lines of code. Pre-trained on 8GB of text. Fine-tuned on SST-2 (67k movie reviews).")
print(f"This is the production reality: you rarely train from scratch.")

In [ ]:
# ============================================================
# SECTION 6: What BERT's Tokenizer Actually Does
# Understanding the input format is critical for production use
# ============================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

test_sentence = "Transfer learning with transformers achieves state of the art results."
encoded = tokenizer(test_sentence, return_tensors="pt")

print("BERT Tokenisation")
print("=" * 55)
print(f"Input:        '{test_sentence}'")
print(f"\nToken IDs:    {encoded['input_ids'][0].tolist()}")
print(f"Tokens:       {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
print(f"Attn mask:    {encoded['attention_mask'][0].tolist()}")
print(f"Sequence len: {encoded['input_ids'].shape[1]}")

print(f"\nKey observations:")
print(f"  [CLS] at position 0: classification token — its final embedding is used for classification")
print(f"  [SEP] at the end:    separator token — marks sentence boundaries")
print(f"  'state' -> 'state' (kept whole)")
print(f"  Long words get split: e.g., 'transformers' may become ['transform', '##ers']")
print(f"  This is WordPiece tokenisation: unknown words split into known subwords")

# Show subword tokenisation on technical ML terms
ml_terms = ["backpropagation", "hyperparameter", "regularization",
            "tokenization", "overfitting", "convolution"]
print(f"\nSubword tokenisation on ML terms:")
for term in ml_terms:
    tokens = tokenizer.tokenize(term)
    print(f"  {term:20s} -> {tokens}")

In [ ]:
# ============================================================
# SECTION 7: Visualise BERT's Attention Heads
# Show what a trained Transformer actually attends to
# ============================================================
from transformers import AutoModel
import torch

model_bert = AutoModel.from_pretrained("distilbert-base-uncased", output_attentions=True)
model_bert.eval()

sentence = "the cat sat on the mat"
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = model_bert(**inputs)

# outputs.attentions: tuple of (num_layers,) each shape (batch, heads, seq_len, seq_len)
# DistilBERT has 6 layers, 12 heads each
attentions = outputs.attentions

print(f"Sentence:         '{sentence}'")
print(f"Tokens:           {tokens}")
print(f"Num layers:       {len(attentions)}")
print(f"Attention shape:  {attentions[0].shape}  (batch, heads, seq_len, seq_len)")

# Visualise attention from layer 3, heads 0-5
layer_idx = 3
attn_layer = attentions[layer_idx][0].detach().numpy()  # (heads, seq_len, seq_len)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for head_idx, ax in enumerate(axes.flat):
    im = ax.imshow(attn_layer[head_idx], cmap='Blues', vmin=0, vmax=attn_layer[head_idx].max())
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.suptitle(f'BERT Attention Patterns: Layer {layer_idx+1}, 6 Heads\n'
             f'Each head learns to attend to different linguistic relationships',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nDifferent heads learn different patterns:")
print("  - Some heads attend strongly to adjacent tokens (local syntax)")
print("  - Some heads learn coreference (linking 'the' to the noun it modifies)")
print("  - Some heads focus on [CLS] or [SEP] tokens")
print("  - Deeper layers tend to capture more semantic relationships")
print("  (Clarber et al., 2019 — 'What Does BERT Look At?')")

## Real World Problem: Context Window Limits in Production

A legal tech company fine-tunes BERT for contract clause classification. BERT has a hard 512-token limit. Most contracts have 5,000-50,000 tokens. The team's solution: truncate every contract to the first 512 tokens. The model ships.

Three months later: the product team reports that high-value contracts are being misclassified at twice the rate of short contracts. The issue: important clauses (indemnification, limitation of liability) are often at the end of contracts, outside the 512-token window.

**What production teams actually do for long documents:**

1. **Sliding window with overlap:** Chunk the document into 512-token windows with 50-100 token overlap. Run the model on each chunk. Aggregate predictions (majority vote or average probabilities).

2. **Hierarchical models:** First model classifies sentences. Second model aggregates sentence-level representations to document level. Used in long-document legal and clinical NLP.

3. **Longformer / BigBird:** Transformers designed for long sequences with sparse attention (not every token attends to every other token). Handle up to 4,096 or 16,384 tokens. These are the production standard for long documents now.

4. **Two-stage retrieval:** Use TF-IDF or BM25 to find the most relevant sections of a long document, then run BERT only on those sections. Fast, interpretable.

**The lesson:** Your model's context window is a design constraint that must be handled explicitly. Truncation is never the right default when the truncated portion contains critical information.

## Interview Corner: MNC-Level Questions

---

**Q1: Why does self-attention scale as O(n²) with sequence length and what are the implications?**

*What they're testing:* Understanding of the core computational bottleneck.

*Answer direction:* In self-attention, every token attends to every other token. For a sequence of n tokens, the attention matrix is n×n. Computing QK^T is O(n² * d_k) in time and O(n²) in memory to store the attention matrix. For n=512 (BERT), this is manageable. For n=16,384 (long documents), the attention matrix alone is 16,384² * 4 bytes ≈ 1GB per layer. This is why long-context Transformers use sparse attention patterns (Longformer's sliding window + global attention) or linear approximations (Performer's kernel trick) to reduce to O(n log n) or O(n).

---

**Q2: What is the difference between encoder-only, decoder-only, and encoder-decoder Transformer architectures? Give an example of each.**

*What they're testing:* Practical knowledge of the model landscape.

*Answer direction:* Encoder-only (BERT): bidirectional self-attention, sees the full input at once. Used for understanding tasks: classification, NER, question answering. Decoder-only (GPT-4, LLaMA): causal self-attention, each token only attends to previous tokens. Used for generation: text generation, code completion, chat. Encoder-decoder (T5, BART): encoder processes the input, decoder generates the output. Used for seq2seq tasks: translation, summarisation, conditional generation. The choice depends on your task: if you need to understand input -> encoder-only. If you need to generate output from scratch -> decoder-only. If you need to transform input to output -> encoder-decoder.

---

**Q3: Why is Layer Normalisation used in Transformers instead of Batch Normalisation?**

*What they're testing:* Understanding of normalisation in sequential models.

*Answer direction:* Batch Normalisation normalises across the batch dimension for each feature. This requires a reasonably large batch and behaves differently at train vs inference time (uses running statistics). In Transformers, sequences have variable lengths and batch sizes are often small. More critically, BatchNorm statistics for NLP don't transfer well: the statistics of the first word in a sentence are very different from the 50th word. Layer Normalisation normalises across the feature dimension for each example independently. It doesn't depend on batch size, works the same at train and inference time, and doesn't conflate statistics across different sequence positions.

---

**Q4: What is the purpose of the residual (skip) connections in the Transformer?**

*What they're testing:* Understanding of training deep networks.

*Answer direction:* Residual connections add the input of a layer to its output: output = x + F(x). They solve two problems. (1) Gradient flow: during backpropagation, the gradient flows through the addition operation unmodified — it doesn't need to pass through the attention or FFN transformation. This prevents vanishing gradients even in 96-layer models like GPT-3. (2) Layer bypass: if a layer learns nothing useful (F(x) ≈ 0), the residual connection means the output is just x — the signal passes through unchanged. Without residuals, useless layers would corrupt the representation. With residuals, the network only needs to learn residual functions (improvements on top of identity), which is an easier optimisation problem.

---

**Q5: You need to classify customer support tickets (avg. 200 words) into 15 categories. Walk me through your modelling approach.**

*What they're testing:* End-to-end applied NLP thinking.

*Answer direction:* Start with distilbert-base-uncased as the backbone: 200 words ≈ 300-350 tokens, comfortably within the 512-token limit, and DistilBERT gives 97% of BERT's quality at 60% of the compute. Add a classification head: [CLS] token embedding -> Linear(768, 15). Fine-tune with AdamW and linear warmup for 3-5 epochs on your labelled tickets. If accuracy is insufficient, switch to bert-base or roberta-base. Evaluate with macro-F1 (not accuracy — ticket categories are probably imbalanced). For deployment: quantise to int8 for 4x faster inference. Log confidence scores in production to detect out-of-distribution tickets (tickets that don't fit any category well should go to human review rather than be auto-classified).

## ML Spotlight

**FlashAttention (Dao et al., Stanford, 2022)**

FlashAttention is an exact, hardware-aware implementation of self-attention that reduces memory from O(n²) to O(n) by tiling the attention computation to fit in GPU SRAM, avoiding slow HBM reads/writes. It's 2-4x faster than standard attention with identical outputs.

It's now integrated into PyTorch via `F.scaled_dot_product_attention` (PyTorch 2.0+) and is used in virtually every production LLM: LLaMA, Mistral, GPT-4 (likely), Gemini.

In PyTorch 2.0+:
```python
# Automatically uses FlashAttention when available
with torch.backends.cuda.sdp_kernel(enable_flash=True):
    output = F.scaled_dot_product_attention(Q, K, V)
```

Paper: [FlashAttention: Fast and Memory-Efficient Exact Attention](https://arxiv.org/abs/2205.14135)

GitHub: https://github.com/Dao-AILab/flash-attention

## Practice Exercise

1. Add a causal mask to the `MultiHeadAttention` class so it only attends to previous tokens (decoder-style). This is the difference between BERT (bidirectional) and GPT (causal):
```python
# Causal mask: upper triangle = 0 (masked), lower = 1 (visible)
mask = torch.tril(torch.ones(seq_len, seq_len))
```

2. Use the HuggingFace pipeline for Named Entity Recognition (NER):
```python
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
result = ner("Google was founded by Larry Page and Sergey Brin in Menlo Park.")
```
Run it on 5 ML-domain sentences and check which entities it detects.

3. Experiment with temperature in text generation using GPT-2:
```python
gen = pipeline("text-generation", model="gpt2")
gen("The most important thing in machine learning is", max_length=50, temperature=0.7)
```

---

**What's Next**

Day 31: Backpropagation and Optimizers — How gradients flow through a neural network, why Adam outperforms SGD in most cases, and what learning rate schedulers actually do to your training dynamics.